# 01a · Exploración del dataset TBX11K

Este notebook es el punto de entrada del proyecto. Antes de entrenar cualquier modelo
necesito entender qué hay en el dataset: cómo están organizadas las imágenes, qué
etiquetas son utilizables, en qué formato vienen los archivos y si hay problemas de
calidad que puedan contaminar los experimentos posteriores.

**El notebook solo lee** `data/raw/TBX11K/`; no escribe ningún archivo.

Preguntas que guían la exploración:
- ¿Qué carpetas e imágenes tienen etiqueta utilizable para TB vs. no-TB?
- ¿Qué formato tienen las imágenes (tamaño, canales, profundidad de bits)?
- ¿Cómo se distribuyen las intensidades por clase y qué implica eso para la
  radiómica de primer orden?
- ¿Hay duplicados o anotaciones vacías que afecten el muestreo del subconjunto?

**Relación con el resto del proyecto**

```
01a (este notebook) ──► 01b (Montgomery-Shenzhen) ──► 02 (segmentación) ──► ...
```
TBX11K es el dataset objetivo: las 1600 imágenes de `subset.csv` son las que
terminarán pasando por los segmentadores y el extractor de características.
Montgomery-Shenzhen (01b) se usa solo para *entrenar* los segmentadores,
porque TBX11K no tiene máscaras pulmonares.

## 1 · Configuración del entorno

In [ ]:
import hashlib
import json
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle

# ── Rutas ──────────────────────────────────────────────────────────────────────
# Busco la raíz del proyecto subiendo desde el directorio actual. Así el notebook
# funciona tanto si lo ejecuto desde notebooks/ como desde la raíz.
ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data" / "raw" / "TBX11K").exists()
)
DATA_DIR = ROOT / "data" / "raw" / "TBX11K"
IMG_DIR  = DATA_DIR / "imgs"
LIST_DIR = DATA_DIR / "lists"
ANNO_DIR = DATA_DIR / "annotations"

# ── Semilla ────────────────────────────────────────────────────────────────────
SEED = 42
rng  = np.random.default_rng(SEED)

# ── Clases con etiqueta pública ────────────────────────────────────────────────
# La etiqueta de cada imagen sale de la carpeta donde está guardada.
# Las carpetas test/ y extra/ no tienen etiqueta utilizable y se excluyen.
CLASSES       = {"tb": "TB", "health": "Sano", "sick": "Enfermo no-TB"}
CLASS_COLORS  = {"tb": "#2a78d6", "health": "#eb6834", "sick": "#1baf7a"}

# ── Estilo de figuras ──────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 110,
    "figure.facecolor": "#fcfcfb",
    "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c9c8c3",
    "axes.labelcolor": "#52514e",
    "axes.titlecolor": "#0b0b0b",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#ecebe7",
    "xtick.color": "#52514e",
    "ytick.color": "#52514e",
    "legend.frameon": False,
})
pd.set_option("display.max_columns", 30)
print("Raíz del proyecto:", ROOT)

## 2 · Estructura del dataset

### 2.1 Carpetas de imágenes

TBX11K organiza sus imágenes en cinco carpetas dentro de `imgs/`:

| Carpeta | Etiqueta | Uso en el proyecto |
|---------|----------|--------------------|
| `tb/`     | TB (activa + latente) | ✅ incluida |
| `health/` | Sano                  | ✅ incluida |
| `sick/`   | Enfermo no-TB         | ✅ incluida |
| `test/`   | Sin etiqueta          | ❌ excluida |
| `extra/`  | Otras fuentes (Montgomery, Shenzhen, DA/DB) | ❌ excluida |

Las listas `TBX11K_train.txt`, `TBX11K_val.txt` y `TBX11K_trainval.txt` dentro
de `lists/` definen la partición oficial de los autores.

### 2.2 Índice de imágenes etiquetadas

In [ ]:
def read_list(name):
    """Lee un archivo de lista y devuelve un conjunto de rutas relativas."""
    return set((LIST_DIR / name).read_text().split())

# Cargo las tres listas oficiales
train_list, val_list, trainval_list = (
    read_list(f"TBX11K_{s}.txt") for s in ["train", "val", "trainval"]
)

# Construyo un índice con todas las imágenes de las tres carpetas etiquetadas.
# Cada fila tiene el identificador de la imagen, su ruta relativa, su clase y
# a qué partición oficial pertenece.
index = pd.DataFrame(
    [
        {"image_id": p.stem, "rel_path": f"{c}/{p.name}", "source_class": c}
        for c in CLASSES
        for p in sorted((IMG_DIR / c).glob("*.png"))
    ]
)
index["label"] = (index["source_class"] == "tb").astype(int)
index["official_split"] = np.where(
    index["rel_path"].isin(train_list), "train",
    np.where(index["rel_path"].isin(val_list), "val", "none")
)

print(f"Total de imágenes etiquetadas: {len(index)}")
print()
display(
    index.groupby(["source_class", "official_split"])
         .size()
         .rename("imágenes")
         .unstack(fill_value=0)
)

Las 8 400 imágenes de `tb/`, `health/` y `sick/` coinciden exactamente con
`TBX11K_trainval.txt` y la partición oficial es aproximadamente 75 % / 25 %,
estratificada por clase.

Para la tarea TB vs. no-TB hay **800 positivos frente a 7 600 negativos**,
un desbalance de 1:9.5 que debo tener en cuenta al evaluar clasificadores.

### 2.3 Anotaciones de lesiones TB (cajas delimitadoras)

In [ ]:
# Recorro todos los XML de anotaciones. Solo las imágenes TB tienen XML;
# las imágenes sanas y enfermas no-TB no vienen anotadas con cajas.
xml_rows, box_rows = [], []
for xml_path in sorted((ANNO_DIR / "xml").glob("*.xml")):
    root    = ET.parse(xml_path).getroot()
    objects = root.findall("object")
    xml_rows.append({
        "image_id":   xml_path.stem,
        "orig_width":  int(root.find("size/width").text),
        "orig_height": int(root.find("size/height").text),
        "n_boxes":     len(objects),
        # Concateno las categorías únicas de esta imagen, ordenadas
        "categories":  " + ".join(
            sorted({o.find("name").text for o in objects})
        ) or "(sin cajas)",
    })
    box_rows += [
        {"image_id": xml_path.stem, "category": o.find("name").text}
        for o in objects
    ]

xml_df = pd.DataFrame(xml_rows)
box_df = pd.DataFrame(box_rows)

# Verifico que haya exactamente un XML por imagen TB y ninguno de más.
tb_ids = set(index.loc[index["label"] == 1, "image_id"])
print("¿Un XML por cada imagen TB?:", set(xml_df["image_id"]) == tb_ids)
print()
print("Distribución de cajas por categoría:")
display(box_df["category"].value_counts().rename("cajas").to_frame())
print()
print("Combinaciones de categorías por imagen:")
display(xml_df["categories"].value_counts().rename("imágenes").to_frame())
print()
print("Cajas por imagen:")
display(
    xml_df["n_boxes"]
    .value_counts().sort_index()
    .rename("imágenes")
    .rename_axis("cajas por imagen")
    .to_frame()
)
print("Imágenes TB sin cajas:", xml_df.loc[xml_df["n_boxes"] == 0, "image_id"].tolist())

Hay dos categorías de lesión:
- **`ActiveTuberculosis`**: TB activa.
- **`ObsoletePulmonaryTuberculosis`**: TB latente (antigua u obsoleta).

La clase TB mezcla ambas; 30 imágenes tienen cajas de los dos tipos.
La imagen `tb1199` es TB pero no tiene cajas de lesión, lo cual no afecta
la clasificación porque su etiqueta sigue siendo TB.

Las anotaciones son **cajas de lesiones**, no máscaras de pulmón. Para la
segmentación pulmonar tengo que usar Montgomery-Shenzhen (notebook 01b).

## 3 · Exploración de imágenes

### 3.1 Formato y estadísticas de intensidad por imagen

In [ ]:
# Recorro las 8 400 imágenes una sola vez. Por cada imagen registro:
#   - formato (forma, tipo, si los canales RGB son idénticos)
#   - estadísticas de intensidad (media, desviación, percentiles, fracción de negro/blanco)
#   - histograma de niveles de gris de 256 celdas
#   - hash MD5 del canal de grises (para detectar duplicados exactos)
# También acumulo la suma y la suma cuadrada de píxeles por clase,
# para calcular la imagen promedio y la desviación estándar espacial en 3.5.

stats_rows  = []
histograms  = np.zeros((len(index), 256), dtype=np.int64)
pix_sum     = {c: np.zeros((512, 512)) for c in CLASSES}
pix_sq      = {c: np.zeros((512, 512)) for c in CLASSES}

for i, row in enumerate(index.itertuples()):
    img = cv2.imread(str(IMG_DIR / row.rel_path), cv2.IMREAD_UNCHANGED)

    # Compruebo si los tres canales son idénticos (para saber si puedo leer
    # en escala de grises sin perder información)
    channels_equal = img.ndim == 2 or (
        np.array_equal(img[..., 0], img[..., 1]) and
        np.array_equal(img[..., 1], img[..., 2])
    )
    # Tomo el primer canal como imagen de grises (verificado arriba)
    gray = img if img.ndim == 2 else img[..., 0]

    histograms[i] = np.bincount(gray.ravel(), minlength=256)
    pix_sum[row.source_class] += gray
    pix_sq[row.source_class]  += gray.astype(np.float64) ** 2

    stats_rows.append({
        "shape":          img.shape,
        "dtype":          str(img.dtype),
        "channels_equal": channels_equal,
        "mean":           gray.mean(),
        "std":            gray.std(),
        "p01":            np.percentile(gray, 1),
        "p99":            np.percentile(gray, 99),
        "frac_black":     (gray == 0).mean(),
        "frac_white":     (gray == 255).mean(),
        "md5":            hashlib.md5(gray.tobytes()).hexdigest(),
    })

stats = pd.concat([index, pd.DataFrame(stats_rows)], axis=1)

print("Formas encontradas:", Counter(stats["shape"]))
print("Tipos de dato:",      Counter(stats["dtype"]))
print("¿Canales RGB idénticos en todas?:", stats["channels_equal"].all())

Todas las imágenes miden 512 × 512 píxeles, son `uint8` y se guardaron en RGB
con los **tres canales idénticos**. Puedo leerlas con `cv2.IMREAD_GRAYSCALE`
sin perder información, lo que simplifica el pipeline y ahorra memoria.

### 3.2 Ejemplos aleatorios por clase

In [ ]:
# Muestro 6 radiografías aleatorias de cada clase para tener una idea visual
# de cómo se ven antes de procesar nada.
n_cols = 6
fig, axes = plt.subplots(len(CLASSES), n_cols,
                          figsize=(2.2 * n_cols, 2.4 * len(CLASSES)))

for r, cls in enumerate(CLASSES):
    ids = rng.choice(
        index.loc[index["source_class"] == cls, "rel_path"], n_cols, replace=False
    )
    for c, rel in enumerate(ids):
        ax = axes[r, c]
        ax.imshow(
            cv2.imread(str(IMG_DIR / rel), cv2.IMREAD_GRAYSCALE),
            cmap="gray", vmin=0, vmax=255
        )
        ax.set_title(Path(rel).stem, fontsize=8, color="#52514e")
        ax.axis("off")
    axes[r, 0].text(
        -40, 256, CLASSES[cls], rotation=90, va="center", ha="right", fontsize=11
    )

fig.suptitle("Radiografías aleatorias por clase", x=0.02, ha="left", fontsize=13)
fig.tight_layout()
plt.show()

### 3.3 Distribución espacial de las lesiones TB

In [ ]:
# Las cajas de lesión vienen en formato COCO (JSON), ya escaladas a 512×512.
# Las uso para ver dónde se ubican las lesiones dentro de la imagen.
coco      = json.loads((ANNO_DIR / "json" / "TBX11K_trainval_only_tb.json").read_text())
cat_names = {c["id"]: c["name"] for c in coco["categories"]}
img_files = {im["id"]: im["file_name"] for im in coco["images"]}

boxes_by_file = defaultdict(list)
for a in coco["annotations"]:
    boxes_by_file[img_files[a["image_id"]]].append(
        (a["bbox"], cat_names[a["category_id"]])
    )

# ── 6 ejemplos con sus cajas ──────────────────────────────────────────────────
BOX_COLORS = {
    "ActiveTuberculosis":         "#eb6834",
    "ObsoletePulmonaryTuberculosis": "#2a78d6",
}
sample = rng.choice(sorted(boxes_by_file), 6, replace=False)
fig, axes = plt.subplots(1, 6, figsize=(14, 2.8))
for ax, fname in zip(axes, sample):
    ax.imshow(
        cv2.imread(str(IMG_DIR / fname), cv2.IMREAD_GRAYSCALE),
        cmap="gray", vmin=0, vmax=255
    )
    for (x, y, w, h), cat in boxes_by_file[fname]:
        ax.add_patch(Rectangle((x, y), w, h, fill=False, lw=2, ec=BOX_COLORS[cat]))
    ax.set_title(Path(fname).stem, fontsize=8, color="#52514e")
    ax.axis("off")

handles = [Rectangle((0, 0), 1, 1, fill=False, lw=2, ec=col) for col in BOX_COLORS.values()]
fig.legend(handles, ["TB activa", "TB latente (obsoleta)"], loc="lower center", ncol=2)
fig.tight_layout(rect=(0, 0.08, 1, 1))
plt.show()

In [ ]:
# Mapa de densidad de lesiones: acumulo cuántas cajas cubren cada píxel.
# El resultado muestra visualmente en qué zonas del pulmón se concentran las lesiones.
heat = np.zeros((512, 512))
for items in boxes_by_file.values():
    for (x, y, w, h), _ in items:
        heat[int(y):int(np.ceil(y + h)), int(x):int(np.ceil(x + w))] += 1

fig, ax = plt.subplots(figsize=(4.2, 4.2))
im = ax.imshow(heat / heat.max(), cmap="Blues", vmin=0, vmax=1)
ax.set_title("Densidad de cajas de lesión TB", loc="left")
ax.axis("off")
fig.colorbar(im, ax=ax, fraction=0.046, label="Frecuencia relativa")
plt.show()

Las lesiones se concentran en los **campos pulmonares superiores**, patrón clásico
de la TB pulmonar activa. Este mapa es útil para diseñar las cajas-prompt de MedSAM:
si las genero a partir de la posición promedio del pulmón, necesito un margen
generoso porque los bordes varían bastante entre pacientes.

### 3.4 Distribución de intensidades por clase

In [ ]:
# Las características radiómicas de primer orden (media, percentiles, entropía…)
# dependen directamente de la distribución de intensidades dentro de la máscara.
# Si las clases ya difieren antes de segmentar, parte de la señal puede venir
# de diferencias de adquisición y no de la patología.

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
panels = [
    ("mean",       "Intensidad media por imagen",     np.linspace(40,  220, 60)),
    ("std",        "Desviación estándar por imagen",   np.linspace(20,  100, 60)),
    ("frac_black", "Fracción de píxeles en 0 (fondo)", np.linspace(0,   0.3, 60)),
]
for ax, (col, title, bins) in zip(axes, panels):
    for cls, name in CLASSES.items():
        ax.hist(
            stats.loc[stats["source_class"] == cls, col],
            bins=bins, density=True,
            histtype="step", lw=2, color=CLASS_COLORS[cls], label=name
        )
    ax.set_title(title, loc="left", fontsize=11)
    ax.set_ylabel("Densidad")
axes[0].legend()
fig.tight_layout()
plt.show()

# Resumen numérico
display(
    stats.groupby("source_class")[["mean", "std", "frac_black"]]
         .agg(["mean", "std"])
         .round(2)
)

La intensidad media difiere entre clases: sanas ≈ 121, TB ≈ 132, enfermas
no-TB ≈ 148. La fracción de fondo negro también varía: ≈ 9,9 % en sanas,
4,8 % en TB y 1,7 % en enfermas no-TB.

Estas diferencias pueden ser confusores: un clasificador sin segmentación
podría aprender a distinguir las clases a partir del fondo y no del pulmón.
La segmentación que extrae solo el interior del pulmón reduce este riesgo.

### 3.5 Imagen promedio y variabilidad espacial por clase

In [ ]:
# La imagen promedio y el mapa de desviación estándar muestran, píxel a píxel,
# dónde cae típicamente el pulmón y cuánto varía su posición entre pacientes.
# Ambas imágenes se calculan con las sumas acumuladas en la sección 3.1,
# sin tener que releer ningún archivo.

fig, axes = plt.subplots(2, len(CLASSES), figsize=(10, 7))
for c, cls in enumerate(CLASSES):
    n        = (index["source_class"] == cls).sum()
    mean_img = pix_sum[cls] / n
    # Uso np.maximum para evitar valores negativos por errores de punto flotante
    std_img  = np.sqrt(np.maximum(pix_sq[cls] / n - mean_img ** 2, 0))

    axes[0, c].imshow(mean_img, cmap="gray", vmin=0, vmax=255)
    axes[0, c].set_title(f"{CLASSES[cls]} — media (n={n})", fontsize=10)
    axes[1, c].imshow(std_img, cmap="magma", vmin=0, vmax=90)
    axes[1, c].set_title(f"{CLASSES[cls]} — desviación estándar", fontsize=10)

for ax in axes.ravel():
    ax.axis("off")
fig.tight_layout()
plt.show()

La imagen promedio muestra que la posición del pulmón es bastante consistente,
pero los bordes varían (valores altos en el mapa de desviación). Esa zona de
alta varianza es exactamente donde un segmentador puede cometer errores y donde
las cajas-prompt de MedSAM necesitan margen extra.

### 3.6 Detección de duplicados exactos

In [ ]:
# Agrupo por hash MD5 del canal de grises. Si dos imágenes tienen el mismo hash
# son idénticas píxel a píxel, aunque los metadatos del PNG difieran.
dup_groups = [
    g["image_id"].tolist()
    for _, g in stats.groupby("md5")
    if len(g) > 1
]
dup_df = pd.DataFrame({"grupo": range(len(dup_groups)), "image_ids": dup_groups})

# Anoto qué clase(s) aparecen en cada grupo de duplicados
id_to_class = index.set_index("image_id")["source_class"]
dup_df["clases"] = dup_df["image_ids"].map(
    lambda ids: " + ".join(sorted({id_to_class[i] for i in ids}))
)

print("Grupos de imágenes idénticas (píxel a píxel):", len(dup_groups))
display(dup_df["clases"].value_counts().rename("grupos").to_frame())
display(dup_df.head(10))

# Cuántos duplicados caen en particiones oficiales distintas (fuga de datos)
split_by_id = index.set_index("image_id")["official_split"]
cross = sum(
    len({split_by_id[i] for i in ids}) > 1
    for ids in dup_groups
)
print(f"Grupos repartidos entre train y val oficiales: {cross}")

In [ ]:
# Visualizo un par duplicado para confirmar que son idénticas y no solo similares
a, b = dup_groups[0]
fig, axes = plt.subplots(1, 2, figsize=(6, 3.2))
for ax, image_id in zip(axes, (a, b)):
    rel = index.set_index("image_id").loc[image_id, "rel_path"]
    ax.imshow(cv2.imread(str(IMG_DIR / rel), cv2.IMREAD_GRAYSCALE), cmap="gray")
    ax.set_title(image_id, fontsize=9)
    ax.axis("off")
fig.suptitle("Ejemplo de par duplicado (idénticos píxel a píxel)", x=0.02, ha="left")
fig.tight_layout()
plt.show()

Hay **126 pares de imágenes idénticas, todos dentro de la clase `sick`**.
De esas, 38 pares quedan repartidos entre el train y el val oficiales,
lo que significa que esas imágenes habrían contaminado la validación.

⇒ Al construir `subset.csv` elimino los duplicados por hash antes de muestrear
las 400 imágenes de `sick`. El módulo `tbseg.data` ya lo hace.

## 4 · Conclusiones e implicaciones para el proyecto

### Datos utilizables
- Hay **8 400 imágenes con etiqueta**: 800 TB, 3 800 sanas y 3 800 enfermas no-TB.
- Las imágenes de `test/` (sin etiquetas públicas) y `extra/` (otras fuentes)
  quedan fuera del subconjunto de trabajo.
- La clase TB mezcla TB activa y latente; 30 imágenes tienen cajas de ambos tipos.

### Formato
- Todas las imágenes miden 512 × 512 píxeles en `uint8`, RGB con los tres
  canales idénticos. Se cargan con `cv2.IMREAD_GRAYSCALE` sin perder información.
- Los autores redimensionaron los originales (tamaños distintos) a cuadrado;
  la relación de aspecto original se perdió.

### Duplicados
- 126 pares idénticos, todos en `sick`. Se eliminan por hash antes de muestrear.

### Diferencias de adquisición (posibles confusores)
- La intensidad media y la fracción de fondo negro difieren entre clases.
  La segmentación pulmonar reduce este riesgo al enfocar el análisis en el
  interior del pulmón.

### Implicaciones para la segmentación
- Las lesiones TB se concentran en los campos superiores del pulmón.
- Los bordes pulmonares varían entre pacientes; las cajas-prompt de MedSAM
  necesitan un margen generoso.